In [1]:
import os
import sys
import pandas as pd
import numpy as np

In [2]:
project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

print(project_root)

d:\Semesters\6th_Sem + Minor\Minor Project\FairHire


In [3]:
resume_df = pd.read_csv("../data/processed/cleaned_resume_dataset.csv")
job_df = pd.read_csv("../data/processed/cleaned_job_dataset.csv")

print("Resume Dataset :", resume_df.shape)
print("Job Dataset :", job_df.shape)

Resume Dataset : (150000, 7)
Job Dataset : (1067, 11)


In [4]:
print(resume_df.columns.tolist())
print(job_df.columns.tolist())

['Role', 'Resume', 'Decision', 'Reason_for_decision', 'Job_Description', 'Clean_Resume', 'Clean_Job_Description']
['JobID', 'Title', 'ExperienceLevel', 'YearsOfExperience', 'Skills', 'Responsibilities', 'Keywords', 'Clean_Skills', 'Clean_Responsibilities', 'Clean_Keywords', 'Clean_Job_Text']


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english"
)

# create Combined_Text if missing by using available cleaned text columns
if "Combined_Text" not in resume_df.columns:
    if "Clean_Resume" in resume_df.columns and "Clean_Job_Description" in resume_df.columns:
        resume_df["Combined_Text"] = resume_df["Clean_Resume"].fillna("") + " " + resume_df["Clean_Job_Description"].fillna("")
    elif "Clean_Resume" in resume_df.columns:
        resume_df["Combined_Text"] = resume_df["Clean_Resume"].fillna("")
    else:
        raise KeyError("Combined_Text not found and no suitable columns to create it.")

X_tfidf = tfidf.fit_transform(resume_df["Combined_Text"])

print("TF-IDF Shape:", X_tfidf.shape)

TF-IDF Shape: (150000, 156)


In [6]:
from src.keyword_match import calculate_keyword_coverage

In [7]:
resume_df["Keyword_Coverage"] = resume_df.apply(
    lambda row: calculate_keyword_coverage(
        row["Clean_Resume"],
        row["Clean_Job_Description"]
    ),
    axis=1
)

print("Keyword Coverage completed.")

Keyword Coverage completed.


In [8]:
resume_df["Keyword_Coverage"].describe()

count    150000.000000
mean         28.772635
std           9.891694
min          12.900000
25%          19.230000
50%          29.630000
75%          37.040000
max          58.620000
Name: Keyword_Coverage, dtype: float64

In [9]:
from src.semantic_similarity import calculate_semantic_similarity

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="cuda"
)

resume_embeddings = model.encode(
    resume_df["Clean_Resume"].fillna("").tolist(),
    batch_size=128,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

job_embeddings = model.encode(
    resume_df["Clean_Job_Description"].fillna("").tolist(),
    batch_size=128,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

resume_df["Semantic_Similarity"] = np.sum(
    resume_embeddings * job_embeddings,
    axis=1
)

print("Semantic Similarity completed.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1172 [00:00<?, ?it/s]

Batches:   0%|          | 0/1172 [00:00<?, ?it/s]

Semantic Similarity completed.


In [11]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(resume_embeddings, "../models/resume_embeddings.pkl")
joblib.dump(job_embeddings, "../models/job_embeddings.pkl")

['../models/job_embeddings.pkl']

In [12]:
resume_df["Semantic_Similarity"].describe()

count    150000.000000
mean          0.534969
std           0.117464
min           0.162944
25%           0.436879
50%           0.542472
75%           0.631560
max           0.859741
Name: Semantic_Similarity, dtype: float64

In [13]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(
    tfidf,
    "../models/tfidf_vectorizer.pkl"
)

print("TF-IDF Vectorizer Saved")

TF-IDF Vectorizer Saved


In [14]:
os.makedirs("../outputs", exist_ok=True)

resume_df.to_pickle("../outputs/resume_df.pkl")

print("=" * 50)
print("Feature engineering completed successfully!")
print("Dataset saved to: ../outputs/resume_df.pkl")
print("Final Shape:", resume_df.shape)
print("=" * 50)

Feature engineering completed successfully!
Dataset saved to: ../outputs/resume_df.pkl
Final Shape: (150000, 10)


In [15]:
resume_df.head()

,Role,Resume,Decision,Reason_for_decision,Job_Description,Clean_Resume,Clean_Job_Description,Combined_Text,Keyword_Coverage,Semantic_Similarity
0,Operations Manager,Operations Manager with 7 years of experience ...,select,Resume matches the required skills and experie...,We are hiring a Operations Manager to support ...,operation manager 7 year experience fintech. a...,hiring operation manager support team e commer...,operation manager 7 year experience fintech. a...,46.43,0.695716
1,Full Stack Engineer,Software Engineer with 5 years of experience i...,select,Resume matches the required skills and experie...,We are hiring a Full Stack Engineer to support...,software engineer 5 year experience gaming. ja...,hiring full stack engineer support team retail...,software engineer 5 year experience gaming. ja...,29.63,0.629071
2,Customer Support Specialist,Sales Representative with 2 years of experienc...,reject,Resume lacks important skills or experience.,We are hiring a Customer Support Specialist to...,sale representative 2 year experience travel. ...,hiring customer support specialist support tea...,sale representative 2 year experience travel. ...,19.23,0.502708
3,Customer Support Specialist,FP&A Analyst with 3 years of experience in Cyb...,reject,Resume lacks important skills or experience.,We are hiring a Customer Support Specialist to...,fp analyst 3 year experience cybersecurity. va...,hiring customer support specialist support tea...,fp analyst 3 year experience cybersecurity. va...,22.22,0.428280
4,Business Development Manager,Project Manager with 2 years of experience in ...,reject,Resume lacks important skills or experience.,We are hiring a Business Development Manager t...,project manager 2 year experience cybersecurit...,hiring business development manager support te...,project manager 2 year experience cybersecurit...,27.59,0.423714


In [16]:
from sklearn.model_selection import train_test_split

# Create train/test splits from the existing TF-IDF features and target labels
y = resume_df["Decision"]
X_train, X_test, y_train, y_test = train_test_split(
	X_tfidf,
	y,
	test_size=0.2,
	random_state=42,
	stratify=y
)

# Use the split matrices directly; no separate scaling step is needed for this sparse TF-IDF setup
X_train_scaled = X_train
X_test_scaled = X_test

joblib.dump(X_train_scaled, "../models/X_train_scaled.pkl")
joblib.dump(X_test_scaled, "../models/X_test_scaled.pkl")

joblib.dump(y_train, "../models/y_train.pkl")
joblib.dump(y_test, "../models/y_test.pkl")

['../models/y_test.pkl']